## MT5-small: Training Notebook

### Imports and Setup

In [21]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import T5Tokenizer, MT5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
import sacrebleu
import sacrebleu
import os
from tqdm import tqdm

### Model Definition

In [22]:
MODEL_NAME = "google/mt5-small"

MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

BATCH_SIZE = 2           # Small batch size for memory
GRAD_ACCUM = 8           # Effective batch = 16
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM

HING_EPOCHS = 25         # Increased from 15 to 25
HING_LR = 2e-4           # Increased learning rate for faster convergence

SPAN_EPOCHS = 25         # Increased from 15 to 25
SPAN_LR = 2e-4           # Increased learning rate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


### Data Loading

In [23]:
print("\nLoading data...")
hing_train = pd.read_csv("data/hinglish_train.csv")
hing_val = pd.read_csv("data/hinglish_val.csv")
hing_test = pd.read_csv("data/hinglish_test.csv")

span_train = pd.read_csv("data/spanglish_train.csv")
span_val = pd.read_csv("data/spanglish_val.csv")
span_test = pd.read_csv("data/spanglish_test.csv")

# Clean data
for df in [hing_train, hing_val, hing_test, span_train, span_val, span_test]:
    df['source'] = df['source'].fillna('').astype(str).str.strip()
    df['target'] = df['target'].fillna('').astype(str).str.strip()
    
print(f"Hinglish: {len(hing_train)} train, {len(hing_val)} val, {len(hing_test)} test")
print(f"Spanglish: {len(span_train)} train, {len(span_val)} val, {len(span_test)} test")



Loading data...
Hinglish: 753 train, 94 val, 95 test
Spanglish: 847 train, 106 val, 106 test


### Preparing Dataset for Training

In [ ]:
# TranslationDataset:
# - Wraps source/target pairs for prefix-based translation (e.g., "translate hinglish to English:")
# - Tokenizes inputs and targets with fixed max lengths
# - Converts padding tokens in labels to -100 for proper loss masking
# - Returns tensors needed for Seq2Seq training (input_ids, attention_mask, labels)

class TranslationDataset(TorchDataset):
    def __init__(self, df, tokenizer, lang_name):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.lang_name = lang_name
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        source = str(self.df.loc[idx, 'source'])
        target = str(self.df.loc[idx, 'target'])
        
        # Add task prefix
        input_text = f"translate {self.lang_name} to English: {source}"
        
        # Tokenize input
        input_encoding = self.tokenizer(
            input_text,
            max_length=MAX_INPUT_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize target
        target_encoding = self.tokenizer(
            target,
            max_length=MAX_TARGET_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Get labels and replace padding with -100
        labels = target_encoding['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': labels
        }

### Full mT5 Training Pipeline:

In [ ]:
# - Training + validation loops with gradient accumulation
# - Beam-search evaluation + BLEU/chrF metrics
# - Warmup scheduler, early stopping, and best-model saving
# - Final test-set evaluation and CSV export of predictions

def train_epoch(model, dataloader, optimizer, scheduler, grad_accum_steps, device):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(dataloader, desc="Training")
    
    for step, batch in enumerate(progress_bar):
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / grad_accum_steps
        total_loss += loss.item() * grad_accum_steps
        
        # Backward pass
        loss.backward()
        
        # Clear intermediate activations to save memory
        del outputs, loss
        
        # Update weights every grad_accum_steps
        if (step + 1) % grad_accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Clear cache periodically
            if (step + 1) % (grad_accum_steps * 10) == 0:
                torch.cuda.empty_cache()
        
        # Update progress bar
        avg_loss = total_loss / (step + 1)
        progress_bar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    # Handle remaining gradients
    if (step + 1) % grad_accum_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    
    torch.cuda.empty_cache()
    return total_loss / len(dataloader)

def evaluate(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Calculate loss
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            
            # Generate predictions
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4,
            )
            
            # Decode
            preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            
            # Decode labels (replace -100 with pad)
            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
            refs = tokenizer.batch_decode(labels_for_decode, skip_special_tokens=True)
            
            all_preds.extend([p.strip() for p in preds])
            all_labels.extend([r.strip() for r in refs])
            
            # Clear memory
            del outputs, generated, labels_for_decode
    
    torch.cuda.empty_cache()
    
    # Calculate metrics
    avg_loss = total_loss / len(dataloader)
    
    # Filter out empty strings
    valid_pairs = [(p, l) for p, l in zip(all_preds, all_labels) if p and l]
    if valid_pairs:
        valid_preds, valid_labels = zip(*valid_pairs)
        bleu = sacrebleu.corpus_bleu(list(valid_preds), [list(valid_labels)]).score
        chrf = sacrebleu.corpus_chrf(list(valid_preds), [list(valid_labels)]).score
    else:
        bleu = 0.0
        chrf = 0.0
    
    # Print sample predictions
    print("\nSample predictions:")
    for i in range(min(3, len(all_preds))):
        print(f"  Pred: {all_preds[i][:80]}")
        print(f"  Gold: {all_labels[i][:80]}")
        print()
    
    return avg_loss, bleu, chrf, all_preds

def train_model(train_df, val_df, test_df, lang_name, epochs, lr, output_dir):
    print(f"\n{'='*80}")
    print(f"TRAINING {lang_name.upper()} MODEL")
    print(f"{'='*80}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Clear any existing memory
    torch.cuda.empty_cache()
    
    # Load tokenizer and model
    tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
    model = MT5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    
    # Enable gradient checkpointing to save memory
    model.gradient_checkpointing_enable()
    
    model.to(device)
    
    print(f"\nModel: {sum(p.numel() for p in model.parameters()):,} parameters")
    
    # Create datasets
    train_dataset = TranslationDataset(train_df, tokenizer, lang_name)
    val_dataset = TranslationDataset(val_df, tokenizer, lang_name)
    test_dataset = TranslationDataset(test_df, tokenizer, lang_name)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # Setup optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01, eps=1e-6)
    
    total_steps = len(train_loader) * epochs // GRAD_ACCUM
    warmup_steps = int(0.1 * total_steps)  # Increased warmup from 5% to 10%
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    print(f"\nTraining setup:")
    print(f"  Epochs: {epochs}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Gradient accumulation: {GRAD_ACCUM}")
    print(f"  Effective batch size: {EFFECTIVE_BATCH_SIZE}")
    print(f"  Learning rate: {lr}")
    print(f"  Total steps: {total_steps}")
    print(f"  Warmup steps: {warmup_steps}")
    
    # Test first batch
    print("\nTesting first batch...")
    test_batch = next(iter(train_loader))
    test_batch = {k: v.to(device) for k, v in test_batch.items()}
    
    with torch.no_grad():
        test_outputs = model(**test_batch)
        print(f"  Test loss: {test_outputs.loss.item():.4f}")
        
        if test_outputs.loss.item() == 0.0:
            print("  ERROR: Loss is 0! Something is wrong with the data.")
            return
        else:
            print("  ✓ Loss computation verified!")
    
    # Training loop
    best_bleu = 0
    patience = 5  # Early stopping patience
    patience_counter = 0
    
    print(f"\n{'='*80}")
    print("Starting training...")
    print(f"{'='*80}\n")
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 80)
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, GRAD_ACCUM, device)
        
        # Evaluate
        val_loss, bleu, chrf, _ = evaluate(model, val_loader, tokenizer, device)
        
        print(f"\nEpoch {epoch + 1} Results:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  BLEU:       {bleu:.2f}")
        print(f"  chrF:       {chrf:.2f}")
        
        # Save best model
        if bleu > best_bleu:
            best_bleu = bleu
            patience_counter = 0
            print(f"  ✓ New best BLEU: {bleu:.2f} (improved by {bleu - best_bleu + bleu:.2f})")
            model.save_pretrained(f"{output_dir}/best_model")
            tokenizer.save_pretrained(f"{output_dir}/best_model")
        else:
            patience_counter += 1
            print(f"  No improvement (patience: {patience_counter}/{patience})")
            
            # Early stopping
            if patience_counter >= patience and epoch > 10:  # Only after 10 epochs
                print(f"\n  Early stopping triggered after {epoch + 1} epochs")
                break
    
    # Test on best model
    print(f"\n{'='*80}")
    print("Evaluating on test set...")
    print(f"{'='*80}")
    
    # Load best model
    model = MT5ForConditionalGeneration.from_pretrained(f"{output_dir}/best_model")
    model.to(device)
    
    test_loss, test_bleu, test_chrf, test_preds = evaluate(model, test_loader, tokenizer, device)
    
    print(f"\nTest Results:")
    print(f"  Loss: {test_loss:.4f}")
    print(f"  BLEU: {test_bleu:.2f}")
    print(f"  chrF: {test_chrf:.2f}")
    
    # Save predictions
    test_df_copy = test_df.copy()
    test_df_copy['prediction'] = test_preds
    test_df_copy.to_csv(f"{output_dir}/test_predictions.csv", index=False)
    print(f"\n✓ Model and predictions saved to {output_dir}")
    
    # Clean up
    del model, optimizer, scheduler
    torch.cuda.empty_cache()
    
    return test_bleu, test_chrf

### Training and Results

In [26]:
# Train Hinglish
hing_bleu, hing_chrf = train_model(
    hing_train, hing_val, hing_test,
    "Hinglish",
    HING_EPOCHS,
    HING_LR,
    "models/mt5_hinglish"
)

# Train Spanglish
span_bleu, span_chrf = train_model(
    span_train, span_val, span_test,
    "Spanglish",
    SPAN_EPOCHS,
    SPAN_LR,
    "models/mt5_spanglish"
)

# Final summary
print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"\nFinal Test Results:")
print(f"  Hinglish  - BLEU: {hing_bleu:.2f}, chrF: {hing_chrf:.2f}")
print(f"  Spanglish - BLEU: {span_bleu:.2f}, chrF: {span_chrf:.2f}")


TRAINING HINGLISH MODEL

Model: 300,176,768 parameters

Training setup:
  Epochs: 25
  Batch size: 2
  Gradient accumulation: 8
  Effective batch size: 16
  Learning rate: 0.0002
  Total steps: 1178
  Warmup steps: 117

Testing first batch...
  Test loss: 38.1670
  ✓ Loss computation verified!

Starting training...


Epoch 1/25
--------------------------------------------------------------------------------


Evaluating: 100%|██████████| 47/47 [00:29<00:00,  1.62it/s]



Sample predictions:
  Pred: <extra_id_0>
  Gold: Comedies seem to be very cliche and there does not seem to be one that I have fa

  Pred: <extra_id_0>.
  Gold: Me either. He did such a good job with avengers. If he had failed their is no wa

  Pred: <extra_id_0>.
  Gold: I do not think I would rewatch this movie.


Epoch 1 Results:
  Train Loss: 24.0271
  Val Loss:   14.9461
  BLEU:       0.12
  chrF:       2.68
  ✓ New best BLEU: 0.12 (improved by 0.12)

Epoch 2/25
--------------------------------------------------------------------------------


Evaluating: 100%|██████████| 47/47 [02:51<00:00,  3.65s/it]



Sample predictions:
  Pred: <extra_id_0> to English: Comedies to English: Comedies to English: Comedies to E
  Gold: Comedies seem to be very cliche and there does not seem to be one that I have fa

  Pred: <extra_id_0> avengers release kartey. a infinity wars release kartey. a infinity
  Gold: Me either. He did such a good job with avengers. If he had failed their is no wa

  Pred: <extra_id_0> to English: I don't think main woh movie phir se dekhunga.
  Gold: I do not think I would rewatch this movie.


Epoch 2 Results:
  Train Loss: 14.7671
  Val Loss:   5.7155
  BLEU:       0.79
  chrF:       10.89
  ✓ New best BLEU: 0.79 (improved by 0.79)

Epoch 3/25
--------------------------------------------------------------------------------


...


Sample predictions:
  Pred: And we decided by a way that was of the proportion of the Concertgebouw with slo
  Gold: We finally settled on a shape that was the proportion of the Concertgebouw with 

  Pred: And what this does is affect payback period, it's not because you can't use sola
  Gold: And all this does is affect the payback period, it doesn't mean that you couldn'

  Pred: And they went down and they looked at their fingers saying, "Okay, what skills t
  Gold: So they pulled back and they were looking at each other, and they were going, "O


Epoch 24 Results:
  Train Loss: 1.0402
  Val Loss:   1.1824
  BLEU:       50.58
  chrF:       69.26
  No improvement (patience: 2/5)

Epoch 25/25
--------------------------------------------------------------------------------


Evaluating: 100%|██████████| 53/53 [01:53<00:00,  2.13s/it]



Sample predictions:
  Pred: And we decided by a way that was of the proportion of the Concertgebouw with slo
  Gold: We finally settled on a shape that was the proportion of the Concertgebouw with 

  Pred: And what this happens is affecting payback period, it's not because you don't wa
  Gold: And all this does is affect the payback period, it doesn't mean that you couldn'

  Pred: And they went down and they looked at their fingers saying, "Okay, what skills t
  Gold: So they pulled back and they were looking at each other, and they were going, "O


Epoch 25 Results:
  Train Loss: 1.0313
  Val Loss:   1.1826
  BLEU:       50.58
  chrF:       69.22
  No improvement (patience: 3/5)

Evaluating on test set...


Evaluating: 100%|██████████| 53/53 [01:49<00:00,  2.07s/it]


Sample predictions:
  Pred: And so, if you can say all this, any of you who know politics think that this is
  Gold: And of course, describing all this, any of you who know politics will think this

  Pred: But then, running in parallel to that, there's a second system that we discovere
  Gold: But then, running in parallel to that is a second system that we've discovered, 

  Pred: And we have created a project we call the CyArk 500 Challenge; which is digitall
  Gold: And we created a project we call the CyArk 500 Challenge -- and that is to digit


Test Results:
  Loss: 1.1805
  BLEU: 52.56
  chrF: 71.04

✓ Model and predictions saved to models/mt5_spanglish

TRAINING COMPLETE!

Final Test Results:
  Hinglish  - BLEU: 14.12, chrF: 34.18
  Spanglish - BLEU: 52.56, chrF: 71.04
